# Project 03: Gibbs Sampling (Random Algorithm)

In [14]:
import random
import numpy as np
#import bamnostic as bs:  delete out line because not needed for FASTA files (most likely leftover from previous BAM)
import seqlogo

#import function for building sequence motif & idenfitying seqs matching to motif
from data_readers import *
from seq_ops import get_seq
from motif_ops import *

In [ ]:
def GibbsMotifFinder (seqs, k, seed=None):
    '''
    Function to find a pfm from a list of strings using a Gibbs sampler
    
    Args: 
        seqs (str list): a list of sequences, not necessarily in same lengths
        k (int): the length of motif to find
        seed (int, default=None): seed for np.random

    Returns:
        pfm (numpy array): dimensions are 4xlength
    '''
    # Use rng to make random samples/selections/numbers
    # Example: randint = rng.integer(1, 10)
    random.seed(seed)
    rng = np.random.default_rng(seed)

    #Motifs will hold the current guessed k-length chunk for each sequence in seqs
    Motifs = []
    for seq in seqs:
        #randomly choose a valid starting position for this sequence's k-length window
        start_pos = rng.integers(0, len(seq) - k + 1)

        #extract the k-length chunk starting at that position
        chunk = seq[start_pos : start_pos + k]

        #save this chunk as the current guess for this sequence
        Motifs.append(chunk)

    for round_num in range(10000):
        #randomly pick which sequence to leave out this round
        #Note:  instructions suggest np.random.randint(), but that uses NumPy's global random state and ignores our seed paramter.  We use rng.integers() instead because its functionally equivalent and keeps results reproducible when a seed is provided.
        i = rng.integers(0, len(seqs))

        #build the consensus from everyone's current guess except sequence i
        other_motifs = []
        for j in range(len(Motifs)):
                if j != i:
                    other_motifs.append(Motifs[j])

        #build the PFM, then the PWM, from every guess except sequence i
        pfm = build_pfm(other_motifs, k)
        pwm = build_pwm(pfm)

        #get full sequence we leave out in this round
        left_out_seq = seqs[i]

        #slide k-length window across every valid position in the left-out sequence
        best_scores = []
        best_chunks = []
        for m in range(len(left_out_seq) - k + 1):
            #get forward strand chunk at this position
            forward_chunk = left_out_seq[m : m + k]

            #get same chunk's reverse complement
            reverse_chunk = reverse_complement(forward_chunk)

            #score both forward and reverse complement chunks against the PWM
            forward_score = score_kmer(forward_chunk, pwm)
            reverse_score = score_kmer(reverse_chunk, pwm)

            #keep whichever strand scored higher and its matching chunk
            if forward_score >= reverse_score:
                best_scores.append(forward_score)
                best_chunks.append(forward_chunk)
            else:
                best_scores.append(reverse_score)
                best_chunks.append(reverse_chunk)

        #convert log2 based scores into positive weights for weighted random choice
        weights = 2 ** np.array(best_scores)

        #randomly pick a new position, weighted so higher scoring positions are more likely, but not guaranteed, to be chosen
        weights = weights / weights.sum()  #normalize so weights add up to 1 (neeeded for probabilities)
        new_position = rng.choice(len(best_chunks), p=weights)  #Note: instructions say to use random.choices() or numpy.random.choice(), but rng.choice() is tied to our seeded rng object, same reason as earlier
           
        #update this sequence's guess to the newly chosen chunk (forward or reverse complement)
        Motifs[i] = best_chunks[new_position]
             
    #after all rounds finish, build one final PFM using every sequence's final guess
    final_pfm = build_pfm(Motifs, k)
    return final_pfm     


In [16]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

#read promoters, store in a list of strings
seq_file = "data/GCF_000009045.1_ASM904v1_genomic.fna"  #had to drop .gz files because no longer compressed
gff_file = "data/GCF_000009045.1_ASM904v1_genomic.gff"

seqs = []

for name, seq in get_fasta(seq_file): # For each entry in our FASTA file
    for gff_entry in get_gff(gff_file): # For each entry in our GFF file
        if gff_entry.type == 'CDS': # If this is a coding sequence
            promoter_seq = get_seq(seq, gff_entry.start, gff_entry.end, gff_entry.strand, 50) # Extract 50 bp as a promoter
    
            """
            Because the gibbs sampling assumption is broken in just using promoters,
            and because it takes very long time to randomly progress through so many
            regions, for this example we will pre-filter for sequences that all contain
            part of the shine-dalgarno motif:
            """
            if "AGGAGG" in promoter_seq:
                seqs.append(promoter_seq)

---
# Driver Program

In [17]:
# Run the gibbs sampler:
promoter_pfm = GibbsMotifFinder(seqs,10 )

# Plot the final pfm that is generated: 
#seqlogo.seqlogo(seqlogo.CompletePm(pfm = promoter_pfm.T))  Note:  commented out because throwing errors

In [18]:
#sanity check our results without needing seqlogo
#print raw PFM and information content to verify function found a real motif.  Higher IC score is more confident non-random pattern
print(promoter_pfm)
print(pfm_ic(promoter_pfm))

[[309 321 314 300 303 328 320 303 317 304]
 [104 106 117 105 108  89 107  87 117 130]
 [185 191 182 224 224 199 202 224 204 191]
 [239 219 224 208 202 221 208 223 199 212]]
0.9324870909392331


---
# NRF1 Driver Program

In [19]:
# Here we test your Gibbs sampler.
# You do not need to edit this or the section below. This is the Driver program

#read promoters, store in a list of strings
nrf1_file="data/nrf1_gibbs.fa" 

nrf1_peaks = []

#Data ingest of nrf1_peaks
for name, seq in get_fasta(nrf1_file):   #FOR each (name, seq) entry returned by get_fasta(nrf1_file)
    nrf1_peaks.append(seq)  #add seq directly to nrf1_peaks (no filtering needed)

# Run the gibbs sampler:
nrf1_pfm = GibbsMotifFinder(nrf1_peaks, 10)

# Plot the final pfm that is generated: 
#seqlogo.seqlogo(seqlogo.CompletePm(pfm = nrf1_pfm.T))  Note: commented out because won't run on my environment

In [ ]:
#sanity check our results without needing seqlogo in NRF1 Driver Program 
#Note: NRF1 has ~90,000 sequences (versus ~800 for promoters), so it will take a long time to run 10,000 rounds.  
#A quick 100 round test showed working code but a low IC score, which is expected for just a few rounds
print(nrf1_pfm)
print(pfm_ic(nrf1_pfm))

[[19648 19448 19420 19570 19633 19622 19692 19434 19393 19336]
 [25197 25650 25488 25696 25514 25337 25549 25458 25609 25622]
 [25760 25466 25488 25400 25432 25606 25584 25438 25430 25537]
 [19456 19497 19665 19395 19482 19496 19236 19731 19629 19566]]
0.12849011595495163
